# Extract/Transform PEPs and PRs
This cell iterates through all Python Enhancement Proposals (PEPs) since the August 1, 2018 date cut-off and the child pull requests. The pipeline then keys these PEPs to pull requests (PRs). The code uses GraphQL to reduce the number of callouts to the Github API.

In [2]:
import json
import time
from datetime import datetime
import requests

# 1. Load Configuration
with open("config.json") as f:
    config = json.load(f)

gh_token = config["github_token"]
headers = {
    "Authorization": f"Bearer {gh_token}",
    "Content-Type": "application/json",
}

REPO_OWNER = "python"
REPO_NAME = "peps"
TARGET_DATE = datetime.fromisoformat("2018-08-01")


def get_matching_peps():
    """Fetch PEPs created on or after August 1, 2018."""
    pep_api_url = "https://peps.python.org/api/peps.json"
    resp = requests.get(pep_api_url)
    resp.raise_for_status()
    peps_data = resp.json()

    matching_peps = {}
    for pep_num_str, pep_info in peps_data.items():
        created_str = pep_info.get("created")
        if created_str:
            try:
                created_dt = datetime.strptime(created_str, "%d-%b-%Y")
                if created_dt >= TARGET_DATE:
                    num_int = int(pep_num_str)
                    matching_peps[num_int] = {
                        "formatted_id": f"pep-{num_int:04d}",
                        "title": pep_info.get("title", ""),
                        "created_date": created_str,
                    }
            except ValueError:
                pass
    return matching_peps


def chunk_list(lst, size):
    """Split a list into smaller chunks."""
    for i in range(0, len(lst), size):
        yield lst[i : i + size]


def fetch_pep_prs_graphql(pep_nums):
    """Fetch PRs for multiple PEPs in a single GraphQL query batch."""
    # Build GraphQL query fields dynamically for each PEP
    query_parts = []
    for pep_num in pep_nums:
        formatted_num = f"{pep_num:04d}"
        # We check both standard paths in repository history
        alias_rst = f"pep_{pep_num}_rst"
        alias_md = f"pep_{pep_num}_md"

        query_parts.append(
            f"""
            {alias_rst}: repository(owner: "{REPO_OWNER}", name: "{REPO_NAME}") {{
              object(expression: "HEAD") {{
                ... on Commit {{
                  history(path: "peps/pep-{formatted_num}.rst", first: 50) {{
                    nodes {{
                      associatedPullRequests(first: 5) {{
                        nodes {{ number }}
                      }}
                    }}
                  }}
                }}
              }}
            }}
            {alias_md}: repository(owner: "{REPO_OWNER}", name: "{REPO_NAME}") {{
              object(expression: "HEAD") {{
                ... on Commit {{
                  history(path: "peps/pep-{formatted_num}.md", first: 50) {{
                    nodes {{
                      associatedPullRequests(first: 5) {{
                        nodes {{ number }}
                      }}
                    }}
                  }}
                }}
              }}
            }}
            """
        )

    gql_query = "query {" + "\n".join(query_parts) + "}"
    
    url = "https://api.github.com/graphql"
    resp = requests.post(url, headers=headers, json={"query": gql_query})
    resp.raise_for_status()
    data = resp.json().get("data", {})

    pep_pr_map = {pep_num: set() for pep_num in pep_nums}

    for pep_num in pep_nums:
        alias_rst = f"pep_{pep_num}_rst"
        alias_md = f"pep_{pep_num}_md"

        for alias in [alias_rst, alias_md]:
            repo_obj = data.get(alias)
            if not repo_obj or not repo_obj.get("object"):
                continue

            history_nodes = repo_obj["object"].get("history", {}).get("nodes", [])
            for commit in history_nodes:
                prs = commit.get("associatedPullRequests", {}).get("nodes", [])
                for pr in prs:
                    pep_pr_map[pep_num].add(pr["number"])

    return {k: sorted(list(v)) for k, v in pep_pr_map.items()}


# Pipeline Execution
start_time = time.time()

matching_peps = get_matching_peps()
pep_numbers = sorted(matching_peps.keys())
print(f"Found {len(matching_peps)} PEPs created on or after August 1, 2018.\n")

pep_pr_map = {}
# Process in batches of 30 PEPs per GraphQL query (~5 total HTTP calls)
print("Executing GraphQL batch queries...")
for chunk in chunk_list(pep_numbers, 30):
    batch_map = fetch_pep_prs_graphql(chunk)
    pep_pr_map.update(batch_map)

# Print Summary Output
print("\n" + "=" * 65)
print(f"{'PEP ID':<12} | {'Created Date':<12} | {'PR Count':<10} | {'PR Numbers'}")
print("=" * 65)

total_prs_found = 0
for pep_num, info in sorted(matching_peps.items()):
    pr_list = pep_pr_map.get(pep_num, [])
    pr_count = len(pr_list)
    total_prs_found += pr_count

    pr_str = ", ".join(f"#{pr}" for pr in pr_list) if pr_list else "None"
    print(
        f"{info['formatted_id']:<12} | {info['created_date']:<12} | {pr_count:<10} | {pr_str}"
    )

elapsed = round(time.time() - start_time, 2)
print("=" * 65)
print(f"Total matching PEPs: {len(matching_peps)}")
print(f"Total PRs mapped:    {total_prs_found}")
print(f"Execution time:      {elapsed} seconds")

Found 269 PEPs created on or after August 1, 2018.

Executing GraphQL batch queries...

PEP ID       | Created Date | PR Count   | PR Numbers
pep-0013     | 16-Dec-2018  | 11         | #3418, #3580, #3582, #3584, #3958, #4081, #4161, #4163, #4664, #4672, #4870
pep-0584     | 01-Mar-2019  | 3          | #3418, #3475, #4081
pep-0585     | 03-Mar-2019  | 3          | #3418, #3821, #4081
pep-0586     | 14-Mar-2019  | 4          | #3418, #3666, #3821, #4081
pep-0587     | 27-Mar-2019  | 3          | #3418, #3475, #4081
pep-0588     | 27-Mar-2019  | 3          | #3418, #3475, #4086
pep-0589     | 20-Mar-2019  | 4          | #3418, #3667, #3821, #4358
pep-0590     | 29-Mar-2019  | 3          | #3418, #3598, #4412
pep-0591     | 15-Mar-2019  | 3          | #3418, #3668, #3821
pep-0592     | 07-May-2019  | 3          | #3418, #3844, #4081
pep-0593     | 26-Apr-2019  | 4          | #3418, #3669, #3821, #4892
pep-0594     | 20-May-2019  | 3          | #3418, #3475, #3778
pep-0595     | 12-May-201

# Load PEPs and PRs
This cell inserts the results of keying PEPs to PRs into a MongoDB collection.

In [4]:
from datetime import datetime
from pymongo import MongoClient
import tiktoken

# 1. Setup MongoDB Connection & Tokenizer
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string)
db = client.consensus
collection = db.pep_prs

tokenizer = tiktoken.get_encoding("cl100k_base")

def store_pep_pr_mapping(pep_num, info, pr_list):
    """Store PEP to PR mapping in MongoDB matching the specified schema pattern."""
    formatted_id = info["formatted_id"]
    title = info["title"]
    pr_str = ", ".join(f"#{pr}" for pr in pr_list) if pr_list else "None"

    # Construct text string for token count tracking
    text = f"PEP {pep_num} ({formatted_id}): {title}. PRs: {pr_str}"
    tokens = tokenizer.encode(text)

    doc = {
        "source": "github_peps",
        "url": f"https://peps.python.org/pep-{pep_num:04d}/",
        "repo": f"{REPO_OWNER}/{REPO_NAME}",
        "type": "pep_pr_mapping",
        "pep_number": pep_num,
        "formatted_id": formatted_id,
        "title": title,
        "created_date": info["created_date"],
        "pr_numbers": pr_list,
        "pr_count": len(pr_list),
        "raw_text": text,
        "tokens": tokens,
        "metadata": {
            "fetched_at": datetime.utcnow().isoformat(),
        },
    }

    # Upsert document using pep_number to safely allow re-running without duplicate keys
    collection.update_one({"pep_number": pep_num}, {"$set": doc}, upsert=True)


# 2. Iterate through mapping results from previous cell and insert into MongoDB
print(f"Inserting {len(matching_peps)} records into MongoDB collection '{collection.name}'...")

inserted_count = 0
for pep_num, info in sorted(matching_peps.items()):
    pr_list = pep_pr_map.get(pep_num, [])
    store_pep_pr_mapping(pep_num, info, pr_list)
    inserted_count += 1

print(f"Successfully stored/updated {inserted_count} documents in MongoDB!")

Inserting 269 records into MongoDB collection 'pep_prs'...


/var/folders/q3/c2bh14p50klc7bpnwh4js8zc0000gn/T/ipykernel_77235/159212683.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at": datetime.utcnow().isoformat(),


Successfully stored/updated 269 documents in MongoDB!


# Extract/Load Comments
This cell fetches comments from each pull request and loads the comments into a document, which contains both keys for both the PR and PEP. No transformation procedure occurs here, as the transformation involves textual coding to be handled in a separate pipeline.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import hashlib
import json
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from pymongo import MongoClient, UpdateOne
import tiktoken


def generate_author_pseudonym(author_username):
    """Generate a consistent SHA256 hash pseudonym for an author username."""
    if not author_username or author_username == "ghost":
        return "user_anonymous"

    # Compute plain SHA256 hash
    hash_bytes = hashlib.sha256(
        author_username.lower().strip().encode("utf-8")
    ).hexdigest()

    # Truncate to 12 hex characters for readable identifiers (e.g. user_8f4a1c09b2e1)
    return f"user_{hash_bytes[:12]}"


# 1. Cap MongoClient pool size to fit within Free Tier (Max 100 connections)
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string, maxPoolSize=10)
db = client.consensus

mappings_collection = db.pep_prs
comments_collection = db.pep_pr_comments

tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Setup GitHub HTTP Session
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {gh_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504, 429])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20, pool_maxsize=20)
session.mount("https://", adapter)

# 3. Load Mappings from MongoDB
print(f"Loading PEP-PR mappings from '{mappings_collection.name}'...")

pr_to_peps = {}
for doc in mappings_collection.find({}, {"pep_number": 1, "pr_numbers": 1}):
    pep_num = doc.get("pep_number")
    pr_list = doc.get("pr_numbers", [])
    for pr_num in pr_list:
        if pr_num not in pr_to_peps:
            pr_to_peps[pr_num] = []
        if pep_num and pep_num not in pr_to_peps[pr_num]:
            pr_to_peps[pr_num].append(pep_num)

all_pr_numbers = sorted(list(pr_to_peps.keys()))
print(f"Found {len(all_pr_numbers)} unique PRs.")


def fetch_paginated_endpoint(url):
    """Fetch paginated items from GitHub API."""
    items = []
    page = 1

    while True:
        resp = session.get(url, params={"per_page": 100, "page": page})

        if resp.status_code == 403:
            reset_time = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
            time.sleep(max(reset_time - int(time.time()), 5))
            continue

        if not resp.ok:
            break

        data = resp.json()
        if not data:
            break

        items.extend(data)
        if "next" not in resp.links:
            break
        page += 1

    return items


def fetch_all_comments_for_pr(pr_number):
    """Fetch review & issue comments for a given PR."""
    review_comments_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls/{pr_number}/comments"
    issue_comments_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/issues/{pr_number}/comments"

    review_comments = fetch_paginated_endpoint(review_comments_url)
    issue_comments = fetch_paginated_endpoint(issue_comments_url)

    processed_comments = []
    for c in review_comments:
        processed_comments.append((c, "pull_request_review_comment"))
    for c in issue_comments:
        processed_comments.append((c, "pull_request_issue_comment"))

    return pr_number, processed_comments


def prepare_bulk_doc(comment_data, pr_number, associated_peps):
    """Format comment into a lightweight, IRB-compliant MongoDB BSON document."""
    comment, comment_type = comment_data
    text = comment.get("body", "") or ""

    # Store token COUNT instead of full array of integers to save 50%+ disk space
    token_count = len(tokenizer.encode(text))

    # Plain SHA256 Pseudonymization
    raw_author = (
        comment.get("user", {}).get("login") if comment.get("user") else "ghost"
    )
    pseudonymized_author = generate_author_pseudonym(raw_author)

    comment_id = comment["id"]

    doc = {
        "source": "github",
        "url": comment.get("html_url", ""),
        "repo": f"{REPO_OWNER}/{REPO_NAME}",
        "type": comment_type,
        "pr_number": pr_number,
        "pep_numbers": associated_peps,
        "author_id": pseudonymized_author,  # Pseudonymized handle
        "created_at": comment.get("created_at"),
        "raw_text": text,
        "token_count": token_count,
        "metadata": {
            "comment_id": comment_id,
            "in_reply_to_id": comment.get("in_reply_to_id"),
        },
    }

    # Single upsert keyed strictly on unique comment_id
    return UpdateOne(
        {"metadata.comment_id": comment_id},
        {"$set": doc},
        upsert=True,
    )


# 4. Fetch comments in parallel
print(f"Fetching review & issue comments for {len(all_pr_numbers)} PRs...")
start_time = time.time()

bulk_operations = []
total_comments = 0

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {
        executor.submit(fetch_all_comments_for_pr, pr_num): pr_num
        for pr_num in all_pr_numbers
    }

    for future in as_completed(futures):
        pr_number, comments = future.result()
        total_comments += len(comments)
        associated_peps = pr_to_peps.get(pr_number, [])
        for c in comments:
            bulk_operations.append(prepare_bulk_doc(c, pr_number, associated_peps))

# 5. Bulk Insert in Batches
if bulk_operations:
    print(f"Executing bulk MongoDB write for {len(bulk_operations)} comments...")

    BATCH_SIZE = 1000
    for i in range(0, len(bulk_operations), BATCH_SIZE):
        batch = bulk_operations[i : i + BATCH_SIZE]
        comments_collection.bulk_write(batch)

elapsed = round(time.time() - start_time, 2)
print(
    f"Done! Processed {len(all_pr_numbers)} PRs, stored {total_comments} comments in {elapsed} seconds."
)

Loading PEP-PR mappings from 'pep_prs'...
Found 1099 unique PRs.
Fetching review & issue comments for 1099 PRs...
Executing bulk MongoDB write for 8248 comments...
Done! Processed 1099 PRs, stored 8248 comments in 100.1 seconds.


# Extract & Load PEP Vote Summaries from Discourse
This cell queries the Python Discourse API (`discuss.python.org`) to locate discussion topics and polls associated with each PEP. It extracts vote tallies (option text, total voters, and vote counts) and upserts the aggregated records into the `pep_vote_summary` collection in MongoDB.

In [8]:
from datetime import datetime, timezone
import hashlib
import json
import re
import time
from pymongo import MongoClient, ReplaceOne
import requests

# 1. Load Configuration
with open("config.json") as f:
    config = json.load(f)

# 2. Setup MongoDB Connection
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string, maxPoolSize=10)
db = client.consensus

pep_prs_collection = db.pep_prs
pep_soft_votes_col = db.pep_soft_votes

# Salt for consistent, one-way pseudonymization across notebook cells
USER_SALT = config.get("user_salt", "pep_consensus_salt_2026")
PEPS_API_URL = "https://peps.python.org/api/peps.json"

# Regex patterns to detect soft vote ratings in post text/HTML
SOFT_VOTE_PATTERN = re.compile(
    r"(?<![a-zA-Z0-9_])([\+\-](?:1|0|0\.5|1\/2|\.5))(?![a-zA-Z0-9_])"
)

# Regex to extract Discourse topic IDs directly from Resolution / Discussion URLs
DISCOURSE_TOPIC_REGEX = re.compile(
    r"discuss\.python\.org/t/(?:[^/]+/)?(\d+)"
)


def pseudonymize_username(username: str) -> str:
    """Generate a consistent SHA-256 pseudonymized identifier for a user."""
    if not username:
        return "anon_unknown"
    salted_str = f"{USER_SALT}:{username.strip().lower()}"
    return f"user_{hashlib.sha256(salted_str.encode('utf-8')).hexdigest()[:12]}"


def fetch_pep_index():
    """Fetch official PEP metadata index from peps.python.org/api/peps.json."""
    try:
        resp = requests.get(
            PEPS_API_URL,
            headers={"User-Agent": "PEP-Vote-Collector/1.0"},
            timeout=15,
        )
        if resp.ok:
            return resp.json()
    except Exception as e:
        print(f"Error fetching PEP API index: {e}")
    return {}


def extract_discourse_topic_id(resolution_val, discourse_val):
    """Extract topic ID if resolution or discourse_topic fields point to discuss.python.org."""
    targets = [str(resolution_val or ""), str(discourse_val or "")]
    for target in targets:
        match = DISCOURSE_TOPIC_REGEX.search(target)
        if match:
            return match.group(1)
    return None


def fetch_topic_soft_votes(topic_id, pep_num, pep_title, resolution_url):
    """Fetch all posts in a Discourse topic and parse soft votes (+1, +0, -0, -1)."""
    if not topic_id:
        return []

    topic_url = f"https://discuss.python.org/t/{topic_id}.json"
    try:
        resp = requests.get(
            topic_url,
            headers={"User-Agent": "PEP-Vote-Collector/1.0"},
            timeout=10,
        )
        if not resp.ok:
            return []

        topic_data = resp.json()
        posts = topic_data.get("post_stream", {}).get("posts", [])
        extracted_vote_docs = []

        for post in posts:
            raw_text = post.get("cooked", "")  # Cooked HTML content
            username = post.get("username", "")

            # Look for matches to soft vote patterns (+1, +0, -0, -1, +1/2, etc.)
            matches = SOFT_VOTE_PATTERN.findall(raw_text)

            if matches:
                # Deduplicate ratings within a single post if repeated, preserving order
                unique_ratings = list(dict.fromkeys(matches))

                for rating in unique_ratings:
                    pseudo_author = pseudonymize_username(username)
                    post_id = post.get("id")

                    # Create a single document per soft vote instance
                    vote_doc = {
                        "_id": f"pep{pep_num}_post{post_id}_{rating.replace('/', '_')}",
                        "pep_number": pep_num,
                        "formatted_id": f"pep-{pep_num:04d}",
                        "title": pep_title,
                        "vote_type": "soft_vote",
                        "rating": rating,  # e.g. "+1", "+0", "-0", "-1"
                        "author_pseudonym": pseudo_author,
                        "resolution_url": resolution_url,
                        "discourse": {
                            "topic_id": topic_id,
                            "post_id": post_id,
                            "post_number": post.get("post_number"),
                            "created_at": post.get("created_at"),
                            "post_url": f"https://discuss.python.org/t/{topic_id}/{post.get('post_number')}",
                        },
                        "metadata": {
                            "extracted_at": datetime.now(
                                timezone.utc
                            ).isoformat(),
                        },
                    }
                    extracted_vote_docs.append(vote_doc)

        return extracted_vote_docs

    except Exception:
        return []


# 3. Load PEP Index & Database PEP List
pep_api_index = fetch_pep_index()
pep_docs = list(pep_prs_collection.find({}, {"pep_number": 1, "title": 1}))

print(f"Loaded {len(pep_docs)} PEP records from MongoDB.")
print(f"Loaded {len(pep_api_index)} PEP records from Official PEP API.")

# Clear existing documents from target collection if re-running
pep_soft_votes_col.delete_many({})

bulk_ops = []
peps_with_soft_votes = set()
total_soft_votes = 0

print(
    "Extracting Resolution Discourse threads and scanning for soft votes (+1, +0, -0, -1)..."
)
for doc in pep_docs:
    pep_num = doc.get("pep_number")
    title = doc.get("title", "")

    # Retrieve official metadata for this PEP from API dictionary
    pep_key = str(pep_num)
    api_meta = pep_api_index.get(pep_key, {})

    resolution_val = api_meta.get("resolution")
    discourse_val = api_meta.get("discourse_topic")

    topic_id = extract_discourse_topic_id(resolution_val, discourse_val)

    if topic_id:
        vote_docs = fetch_topic_soft_votes(
            topic_id, pep_num, title, resolution_val
        )

        if vote_docs:
            peps_with_soft_votes.add(pep_num)
            total_soft_votes += len(vote_docs)
            for v_doc in vote_docs:
                bulk_ops.append(
                    ReplaceOne({"_id": v_doc["_id"]}, v_doc, upsert=True)
                )

        time.sleep(0.15)  # Rate limiting delay

# 4. Bulk Write Output
if bulk_ops:
    print(
        f"Inserting {len(bulk_ops)} soft vote documents into '{pep_soft_votes_col.name}'..."
    )
    pep_soft_votes_col.bulk_write(bulk_ops)
else:
    print("No soft votes found in Resolution threads.")

print("=" * 65)
print(f"Total PEPs processed:                   {len(pep_docs)}")
print(f"PEPs with detected Resolution soft votes:{len(peps_with_soft_votes)}")
print(f"Total vote documents created:            {total_soft_votes}")
print(f"Collection update complete:              '{pep_soft_votes_col.name}'")

Loaded 269 PEP records from MongoDB.
Loaded 741 PEP records from Official PEP API.
Extracting Resolution Discourse threads and scanning for soft votes (+1, +0, -0, -1)...
Inserting 43 soft vote documents into 'pep_soft_votes'...
Total PEPs processed:                   269
PEPs with detected Resolution soft votes:28
Total vote documents created:            43
Collection update complete:              'pep_soft_votes'
